<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://www.sap.com/dam/application/shared/logos/sap-logo-svg.svg" alt="SAP" width="100">
</div>

# Agentic JIT Supply Chain Mitigation
## Part 2: From Prediction to Autonomous Action

### The Evolution: From Scoring to Reasoning

In Part 1, you built a **deterministic pipeline**: predict delay → classify risk → lookup alternatives. The workflow is fixed—it always follows the same path.

But real supply chain decisions are rarely that simple:

- *"We have 5 high-risk POs this week. Which one should we address first?"*
- *"VENDOR_A has the best price, but they're at 95% capacity. Should we split the order?"*
- *"This is our third incident with VENDOR_C this quarter. Should we recommend a strategic supplier review?"*

These questions require **adaptive reasoning**—where the next step depends on what we learn along the way. This is where **agentic AI** adds value.

---

### Workshop Roadmap

| Section | Time | Outcome |
|---------|------|--------|
| **Setup** (Steps 2.1-2.2) | 10 min | Configure agent environment |
| **Agent Tools** (Steps 2.3-2.4) | 15 min | Define business capability tools |
| **ReAct Loop** (Steps 2.5-2.6) | 20 min | Build the reasoning engine |
| **Run & Trace** (Steps 2.7-2.8) | 15 min | Execute and observe agent behavior |
| **Checkpoint** | 10 min | Exercise and reflection |

---

### Learning Objectives

By the end of Part 2, you will be able to:
- Explain when agentic orchestration adds value over deterministic workflows
- Implement a ReAct (Reason + Act) agent using SAP Gen AI Hub
- Design tool contracts that expose business capabilities to an LLM
- Apply human-in-the-loop governance for high-stakes decisions

## When to Use an Agent (and When Not To)

### Decision Framework

| Dimension | Deterministic (Part 1) | Agentic (Part 2) |
|-----------|------------------------|------------------|
| **Workflow** | Fixed steps, known in advance | Adaptive, depends on findings |
| **Decisions** | Rule-based thresholds | Multi-criteria reasoning |
| **Data Sources** | Single query per step | May need to explore multiple sources |
| **Output** | Structured result | Recommendation with rationale |
| **Latency** | 1-2 API calls | N tool calls + N LLM calls |
| **Governance** | Simple audit trail | Full reasoning trace + approval gate |

### This Workshop's Agentic Scenario

**Question:** *"Review the high-risk POs for this week and recommend a prioritized mitigation plan."*

Why this requires an agent:
1. Must assess multiple POs, not just one
2. Must reason about relative priority (which risk is highest?)
3. May need to check supplier capacity before recommending
4. Must synthesize findings into a coherent recommendation

The agent's workflow will adapt based on what it discovers.

---

## Part 1: Setup & Configuration

### Step 2.1 - Install and Import Dependencies

We reuse the same SAP SDK from Part 1, plus add orchestration capabilities.

In [ ]:
%pip install generative-ai-hub-sdk==4.12.4 --quiet
%pip install pandas python-dotenv requests --quiet

print("Packages installed.")

In [ ]:
import json
import os
import time
from typing import Any, Dict, List, Tuple
from datetime import datetime

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display, HTML

from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.template import Template
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage

# --- Global Constants ---
MODEL_NAME = "anthropic--claude-4.5-sonnet"
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 60

# --- Risk Tier Thresholds ---
RISK_THRESHOLD_AMBER = 1.0
RISK_THRESHOLD_RED = 3.0

# --- Cost Parameters ---
LINE_DOWN_COST_PER_HOUR = 15000

# --- Tool Name Constants ---
TOOL_ASSESS_PO_RISK = "assess_po_risk"
TOOL_FIND_ALTERNATIVES = "find_alternative_suppliers"
TOOL_CHECK_SUPPLIER_CAPACITY = "check_supplier_capacity"
TOOL_GENERATE_PROPOSAL = "generate_mitigation_proposal"

# --- Agent Response Types ---
ACTION_TYPE = "action"
FINAL_TYPE = "final"
RETRY_TYPE = "retry"

print("Imports and constants loaded.")

### Step 2.2 - Load Configuration and Data

In [ ]:
# Load environment
dotenv_path = find_dotenv()
load_dotenv(dotenv_path=dotenv_path, override=True)

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
ORCH_DEPLOYMENT_URL = os.getenv("ORCH_DEPLOYMENT_URL")

# Check required variables
required = {
    "AICORE_AUTH_URL": AICORE_AUTH_URL,
    "AICORE_CLIENT_ID": AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "AICORE_BASE_URL": AICORE_BASE_URL,
    "AICORE_RESOURCE_GROUP": AICORE_RESOURCE_GROUP,
    "ORCH_DEPLOYMENT_URL": ORCH_DEPLOYMENT_URL,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise EnvironmentError(f"Missing required .env variables: {missing}")

# Initialize orchestration service
orchestration_service = OrchestrationService(api_url=ORCH_DEPLOYMENT_URL)

print(f"Loaded .env from: {dotenv_path}")
print(f"Resource group: {AICORE_RESOURCE_GROUP}")
print(f"Model: {MODEL_NAME}")
print("OrchestrationService initialized.")

In [ ]:
# Load datasets
historical_df = pd.read_csv("data/historical_po_data.csv")
prediction_df = pd.read_csv("data/new_po_prediction.csv")
alt_supplier_df = pd.read_csv("data/alt_supplier_table.csv")

print(f"Loaded {len(historical_df)} historical POs")
print(f"Loaded {len(alt_supplier_df)} alternative supplier records")

---

## Part 2: Define Agent Tools

Tools are the agent's interface to business capabilities. Each tool:
- Has a clear, single responsibility
- Returns structured data the agent can reason about
- Includes source attribution for auditability

### Step 2.3 - Define Business Tools

We create four tools that expose supply chain capabilities:

| Tool | Purpose | When Agent Uses It |
|------|---------|-------------------|
| `assess_po_risk` | Predict delay for a specific PO | First step - understand the risk |
| `find_alternative_suppliers` | Get backup options for a material | When risk is elevated |
| `check_supplier_capacity` | Verify a supplier can fulfill order | Before recommending alternative |
| `generate_mitigation_proposal` | Create structured recommendation | Final synthesis |

In [ ]:
# --- Helper Functions (reused from Part 1) ---

def derive_risk_tier(delay_days: float) -> str:
    """Classify delay into business risk tiers."""
    if delay_days < RISK_THRESHOLD_AMBER:
        return "Green"
    elif delay_days <= RISK_THRESHOLD_RED:
        return "Amber"
    return "Red"


def mock_rpt1_predict(row: pd.Series) -> float:
    """Mock prediction function (same as Part 1)."""
    vendor = row.get("Vendor_ID", "VENDOR_A")
    month = int(row.get("Order_Month", 1))
    qty = int(row.get("Order_Quantity", 1000))
    material_group = row.get("Material_Group", "")
    incoterms = row.get("Incoterms", "CIF")

    delay = 0.0
    if vendor == "VENDOR_A":
        delay += 0.5
    elif vendor == "VENDOR_B":
        delay += 1.8
    else:
        delay += 2.6

    if vendor == "VENDOR_B" and month in [10, 11, 12]:
        delay += 1.5
    if vendor == "VENDOR_C" and qty > 1500:
        delay += 1.2
    if material_group in ["Control Chip", "Optical Module"]:
        delay += 0.3
    if incoterms == "FOB":
        delay += 0.2
    elif incoterms == "DAP":
        delay -= 0.2

    return round(max(delay, 0), 1)


def estimate_avoided_loss(delay_days: float, is_critical: bool) -> int:
    """Estimate business value of mitigation."""
    if not is_critical or delay_days < RISK_THRESHOLD_RED:
        return 0
    disruption_hours = min(delay_days * 4, 16)
    return int(disruption_hours * LINE_DOWN_COST_PER_HOUR)


print("Helper functions ready.")

In [ ]:
# --- Agent Tools ---

def assess_po_risk(po_id: str = None, vendor_id: str = None, material_id: str = None,
                   quantity: int = None, order_month: int = None) -> Dict[str, Any]:
    """
    Assess delivery delay risk for a purchase order.
    
    Args:
        po_id: Specific PO to assess (optional)
        vendor_id: Vendor for new assessment (if no po_id)
        material_id: Material for new assessment
        quantity: Order quantity
        order_month: Month of order (1-12)
    
    Returns:
        Risk assessment with predicted delay, tier, and business impact
    """
    # If PO ID provided, look it up
    if po_id:
        if po_id in prediction_df["PO_ID"].values:
            row = prediction_df[prediction_df["PO_ID"] == po_id].iloc[0]
        elif po_id in historical_df["PO_ID"].values:
            row = historical_df[historical_df["PO_ID"] == po_id].iloc[0]
        else:
            return {"error": f"PO {po_id} not found"}
    else:
        # Create synthetic assessment
        if not all([vendor_id, material_id]):
            return {"error": "Provide either po_id or (vendor_id, material_id)"}
        
        # Get vendor profile
        vendor_data = historical_df[historical_df["Vendor_ID"] == vendor_id]
        if len(vendor_data) == 0:
            return {"error": f"Vendor {vendor_id} not found"}
        
        # Get material info
        mat_data = historical_df[historical_df["Material_ID"] == material_id]
        if len(mat_data) == 0:
            return {"error": f"Material {material_id} not found"}
        
        row = pd.Series({
            "PO_ID": "SYNTHETIC",
            "Vendor_ID": vendor_id,
            "Vendor_Country": vendor_data["Vendor_Country"].iloc[0],
            "Vendor_OTIF_Percent": vendor_data["Vendor_OTIF_Percent"].iloc[0],
            "Material_ID": material_id,
            "Material_Group": mat_data["Material_Group"].iloc[0],
            "Criticality_Flag": mat_data["Criticality_Flag"].iloc[0],
            "Order_Quantity": quantity or 1000,
            "Order_Month": order_month or datetime.now().month,
            "Incoterms": "FOB"
        })
    
    # Predict delay
    predicted_delay = mock_rpt1_predict(row)
    risk_tier = derive_risk_tier(predicted_delay)
    is_critical = row.get("Criticality_Flag", "No") == "Yes"
    avoided_loss = estimate_avoided_loss(predicted_delay, is_critical)
    
    return {
        "tool": TOOL_ASSESS_PO_RISK,
        "po_id": row.get("PO_ID", po_id),
        "vendor_id": row["Vendor_ID"],
        "vendor_country": row.get("Vendor_Country", "Unknown"),
        "material_id": row.get("Material_ID", material_id),
        "material_group": row.get("Material_Group", "Unknown"),
        "is_critical": is_critical,
        "quantity": int(row.get("Order_Quantity", quantity or 0)),
        "predicted_delay_days": predicted_delay,
        "risk_tier": risk_tier,
        "potential_loss_if_delayed": avoided_loss,
        "recommendation": (
            "Immediate mitigation required" if risk_tier == "Red" and is_critical
            else "Monitor closely" if risk_tier in ["Red", "Amber"]
            else "Standard monitoring"
        )
    }


def find_alternative_suppliers(material_id: str, exclude_vendor: str = None) -> Dict[str, Any]:
    """
    Find alternative suppliers for a material.
    
    Args:
        material_id: The material to find alternatives for
        exclude_vendor: Vendor to exclude (typically current supplier)
    
    Returns:
        List of alternative suppliers with lead times, prices, and stock
    """
    candidates = alt_supplier_df[alt_supplier_df["Material_ID"] == material_id].copy()
    
    if exclude_vendor:
        candidates = candidates[candidates["Alt_Vendor"] != exclude_vendor]
    
    if len(candidates) == 0:
        return {
            "tool": TOOL_FIND_ALTERNATIVES,
            "material_id": material_id,
            "alternatives_found": 0,
            "message": "No alternative suppliers found"
        }
    
    # Sort by lead time, then price
    candidates = candidates.sort_values(["Lead_Time_Days", "Indicative_Unit_Price"])
    
    alternatives = []
    for _, row in candidates.iterrows():
        alternatives.append({
            "vendor_id": row["Alt_Vendor"],
            "country": row["Alt_Vendor_Country"],
            "lead_time_days": int(row["Lead_Time_Days"]),
            "unit_price": float(row["Indicative_Unit_Price"]),
            "available_stock": int(row["Current_Available_Stock"])
        })
    
    return {
        "tool": TOOL_FIND_ALTERNATIVES,
        "material_id": material_id,
        "excluded_vendor": exclude_vendor,
        "alternatives_found": len(alternatives),
        "alternatives": alternatives,
        "best_option": alternatives[0] if alternatives else None
    }


def check_supplier_capacity(vendor_id: str, material_id: str, 
                            required_quantity: int) -> Dict[str, Any]:
    """
    Check if a supplier can fulfill a specific order quantity.
    
    Args:
        vendor_id: The supplier to check
        material_id: The material needed
        required_quantity: How many units are needed
    
    Returns:
        Capacity assessment with availability and fulfillment status
    """
    record = alt_supplier_df[
        (alt_supplier_df["Alt_Vendor"] == vendor_id) &
        (alt_supplier_df["Material_ID"] == material_id)
    ]
    
    if len(record) == 0:
        return {
            "tool": TOOL_CHECK_SUPPLIER_CAPACITY,
            "vendor_id": vendor_id,
            "material_id": material_id,
            "error": "Supplier/material combination not found"
        }
    
    row = record.iloc[0]
    available = int(row["Current_Available_Stock"])
    can_fulfill = available >= required_quantity
    
    return {
        "tool": TOOL_CHECK_SUPPLIER_CAPACITY,
        "vendor_id": vendor_id,
        "material_id": material_id,
        "required_quantity": required_quantity,
        "available_stock": available,
        "can_fulfill_fully": can_fulfill,
        "shortfall": max(0, required_quantity - available),
        "lead_time_days": int(row["Lead_Time_Days"]),
        "unit_price": float(row["Indicative_Unit_Price"])
    }


def generate_mitigation_proposal(po_id: str, current_vendor: str, 
                                  recommended_vendor: str, material_id: str,
                                  quantity: int, reason: str) -> Dict[str, Any]:
    """
    Generate a formal mitigation proposal document.
    
    Args:
        po_id: The PO being mitigated
        current_vendor: The original supplier
        recommended_vendor: The proposed alternative
        material_id: Material in question
        quantity: Order quantity
        reason: Business justification
    
    Returns:
        Structured proposal ready for approval workflow
    """
    # Get supplier details
    current_info = alt_supplier_df[
        (alt_supplier_df["Alt_Vendor"] == current_vendor) &
        (alt_supplier_df["Material_ID"] == material_id)
    ]
    
    recommended_info = alt_supplier_df[
        (alt_supplier_df["Alt_Vendor"] == recommended_vendor) &
        (alt_supplier_df["Material_ID"] == material_id)
    ]
    
    if len(recommended_info) == 0:
        return {"error": f"Recommended vendor {recommended_vendor} not found for {material_id}"}
    
    rec = recommended_info.iloc[0]
    
    # Calculate financial impact
    if len(current_info) > 0:
        curr = current_info.iloc[0]
        current_price = float(curr["Indicative_Unit_Price"])
    else:
        current_price = float(rec["Indicative_Unit_Price"])  # Use recommended as baseline
    
    new_price = float(rec["Indicative_Unit_Price"])
    price_delta = (new_price - current_price) * quantity
    
    return {
        "tool": TOOL_GENERATE_PROPOSAL,
        "proposal_id": f"MIT-{po_id}-{datetime.now().strftime('%Y%m%d%H%M')}",
        "generated_at": datetime.now().isoformat(),
        "status": "PENDING_APPROVAL",
        "original_order": {
            "po_id": po_id,
            "vendor": current_vendor,
            "material": material_id,
            "quantity": quantity
        },
        "proposed_change": {
            "new_vendor": recommended_vendor,
            "vendor_country": rec["Alt_Vendor_Country"],
            "lead_time_days": int(rec["Lead_Time_Days"]),
            "unit_price": new_price,
            "available_stock": int(rec["Current_Available_Stock"])
        },
        "financial_impact": {
            "price_difference_total": round(price_delta, 2),
            "is_cost_increase": price_delta > 0
        },
        "justification": reason,
        "approver_required": "Supply Chain Manager",
        "next_steps": [
            "Review proposal and financial impact",
            "Approve or reject in SAP workflow",
            "If approved, system will create purchase requisition"
        ]
    }


# Tool registry
TOOLS = {
    TOOL_ASSESS_PO_RISK: assess_po_risk,
    TOOL_FIND_ALTERNATIVES: find_alternative_suppliers,
    TOOL_CHECK_SUPPLIER_CAPACITY: check_supplier_capacity,
    TOOL_GENERATE_PROPOSAL: generate_mitigation_proposal,
}

print(f"Agent tools registered: {list(TOOLS.keys())}")

### Step 2.4 - Test the Tools

Before building the agent, let's verify each tool works correctly.

In [ ]:
# Test assess_po_risk
print("Testing assess_po_risk:")
risk_result = assess_po_risk(po_id="PO200001")
print(json.dumps(risk_result, indent=2))

print("\n" + "="*60 + "\n")

# Test find_alternative_suppliers
print("Testing find_alternative_suppliers:")
alt_result = find_alternative_suppliers("MAT-1002", exclude_vendor="VENDOR_C")
print(json.dumps(alt_result, indent=2))

print("\n" + "="*60 + "\n")

# Test check_supplier_capacity
print("Testing check_supplier_capacity:")
cap_result = check_supplier_capacity("VENDOR_A", "MAT-1002", 2200)
print(json.dumps(cap_result, indent=2))

---

## Part 3: Build the ReAct Agent

ReAct = **Rea**soning + **Act**ing

The agent follows this loop:
1. **Think**: Analyze the current situation
2. **Act**: Choose and execute a tool
3. **Observe**: Process the tool result
4. **Repeat** until ready to provide final answer

### Step 2.5 - Define Agent Prompt and Parser

The system prompt instructs the agent on:
- Its role and capabilities
- Available tools and when to use them
- Required output format (JSON)

In [ ]:
AGENT_SYSTEM_PROMPT = """
You are a Supply Chain Risk Management Agent for BestRun Technologies.

Your role is to:
1. Assess delivery risk for purchase orders
2. Identify when mitigation is needed
3. Find and evaluate alternative suppliers
4. Generate actionable mitigation proposals

RULES:
- Always use tools to gather data. Do not make up information.
- Think step by step about what information you need.
- For high-risk POs (Red tier + critical material), always propose mitigation.
- Return ONLY valid JSON. No markdown, no extra text.

AVAILABLE TOOLS:
1. assess_po_risk - Predict delay risk for a PO
   Args: {"po_id": "PO200001"} OR {"vendor_id": "VENDOR_A", "material_id": "MAT-1001", "quantity": 1000}

2. find_alternative_suppliers - Find backup suppliers for a material
   Args: {"material_id": "MAT-1002", "exclude_vendor": "VENDOR_C"}

3. check_supplier_capacity - Verify supplier can fulfill order
   Args: {"vendor_id": "VENDOR_A", "material_id": "MAT-1002", "required_quantity": 2200}

4. generate_mitigation_proposal - Create formal proposal document
   Args: {"po_id": "...", "current_vendor": "...", "recommended_vendor": "...", 
          "material_id": "...", "quantity": ..., "reason": "..."}

OUTPUT FORMAT:

For tool calls:
{"type": "action", "thought": "why I need this", "tool": "tool_name", "args": {...}}

For final answer:
{"type": "final", "thought": "summary of analysis", "answer": "recommendation", "confidence": "high|medium|low"}
""".strip()


def extract_json_from_response(text: str) -> Dict[str, Any]:
    """Extract the first valid JSON object from LLM response."""
    # Find first { and matching }
    start = text.find("{")
    if start == -1:
        return {"type": RETRY_TYPE, "error": "No JSON found"}
    
    depth = 0
    in_string = False
    escape = False
    
    for i in range(start, len(text)):
        ch = text[i]
        
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except json.JSONDecodeError as e:
                    return {"type": RETRY_TYPE, "error": f"Invalid JSON: {e}"}
    
    return {"type": RETRY_TYPE, "error": "Unclosed JSON"}


print("Agent prompt and parser ready.")

### Step 2.6 - Implement the ReAct Loop

This is the core agent runtime:
- Maintains conversation history
- Executes tools and feeds results back
- Includes guardrails (min tools, max steps)
- Captures full trace for observability

In [ ]:
def execute_tool(tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    """Execute an agent tool by name."""
    if tool_name not in TOOLS:
        return {"error": f"Unknown tool: {tool_name}. Available: {list(TOOLS.keys())}"}
    
    try:
        return TOOLS[tool_name](**args)
    except TypeError as e:
        return {"error": f"Invalid arguments for {tool_name}: {e}"}
    except Exception as e:
        return {"error": f"Tool {tool_name} failed: {e}"}


def run_supply_chain_agent(
    task: str,
    min_tools: int = 1,
    max_steps: int = 8,
    require_approval: bool = True,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run the supply chain risk management agent.
    
    Args:
        task: The task/question for the agent
        min_tools: Minimum tool calls before final answer
        max_steps: Maximum reasoning steps
        require_approval: Add human-in-the-loop gate
        verbose: Print step-by-step progress
    
    Returns:
        Agent result with answer, trace, and approval state
    """
    messages = [
        SystemMessage(AGENT_SYSTEM_PROMPT),
        UserMessage(task)
    ]
    
    tool_history = []
    trace = []
    
    if verbose:
        print("=" * 70)
        print("SUPPLY CHAIN AGENT")
        print(f"Task: {task}")
        print("=" * 70)
    
    for step in range(1, max_steps + 1):
        step_start = time.time()
        
        # Get agent response
        response = orchestration_service.run(
            config=OrchestrationConfig(
                template=Template(messages=messages),
                llm=LLM(name=MODEL_NAME)
            )
        )
        
        raw = response.orchestration_result.choices[0].message.content
        parsed = extract_json_from_response(raw)
        elapsed_ms = int((time.time() - step_start) * 1000)
        
        response_type = parsed.get("type", "unknown")
        thought = parsed.get("thought", "")
        
        # Handle action (tool call)
        if response_type == ACTION_TYPE:
            tool_name = parsed.get("tool")
            args = parsed.get("args", {})
            
            tool_result = execute_tool(tool_name, args)
            tool_history.append({"tool": tool_name, "args": args})
            
            trace.append({
                "step": step,
                "type": "action",
                "thought": thought,
                "tool": tool_name,
                "args": args,
                "latency_ms": elapsed_ms
            })
            
            if verbose:
                print(f"\nStep {step}: ACTION -> {tool_name}")
                print(f"  Thought: {thought[:80]}..." if len(thought) > 80 else f"  Thought: {thought}")
                print(f"  Args: {args}")
            
            # Add result to conversation
            messages.append(UserMessage(
                f"Tool result:\n{json.dumps(tool_result, indent=2)}\n\n"
                f"Tools used so far: {len(tool_history)}. Continue analysis or provide final answer."
            ))
            continue
        
        # Handle final answer
        if response_type == FINAL_TYPE:
            # Guardrail: ensure minimum tool usage
            if len(tool_history) < min_tools:
                trace.append({
                    "step": step,
                    "type": "guardrail",
                    "thought": f"Min tools not met ({len(tool_history)}/{min_tools})",
                    "latency_ms": elapsed_ms
                })
                
                if verbose:
                    print(f"\nStep {step}: GUARDRAIL - need {min_tools - len(tool_history)} more tool calls")
                
                messages.append(UserMessage(
                    f"You've only used {len(tool_history)} tools. "
                    f"Please gather more data (minimum {min_tools} tool calls) before concluding."
                ))
                continue
            
            answer = parsed.get("answer", "")
            confidence = parsed.get("confidence", "medium")
            
            trace.append({
                "step": step,
                "type": "final",
                "thought": thought,
                "latency_ms": elapsed_ms
            })
            
            if verbose:
                print(f"\nStep {step}: FINAL ANSWER")
                print(f"  Confidence: {confidence}")
            
            return {
                "task": task,
                "answer": answer,
                "confidence": confidence,
                "tool_history": tool_history,
                "trace": trace,
                "steps_taken": step,
                "approval_state": "pending_human_approval" if require_approval else "auto_approved"
            }
        
        # Handle retry (parse error)
        trace.append({
            "step": step,
            "type": "retry",
            "error": parsed.get("error", "unknown"),
            "latency_ms": elapsed_ms
        })
        
        if verbose:
            print(f"\nStep {step}: RETRY - {parsed.get('error', 'format error')}")
        
        messages.append(UserMessage(
            "Invalid response format. Return ONLY a JSON object with 'type' field."
        ))
    
    # Max steps reached
    if verbose:
        print(f"\nMax steps ({max_steps}) reached without conclusion.")
    
    return {
        "task": task,
        "answer": "Agent reached maximum steps without final answer.",
        "confidence": "low",
        "tool_history": tool_history,
        "trace": trace,
        "steps_taken": max_steps,
        "approval_state": "incomplete"
    }


print("Agent runtime ready.")

---

## Part 4: Run the Agent

### Step 2.7 - Execute Agent Scenario

Let's run the agent on a realistic supply chain task.

In [ ]:
# Define the task
agent_task = """
Assess the risk for PO PO200001 which is a large order of Control Chips from VENDOR_C.
If the risk is elevated, find alternative suppliers and create a mitigation proposal.
Consider that this is a critical component for our production line.
"""

# Run the agent
agent_result = run_supply_chain_agent(
    task=agent_task,
    min_tools=2,
    max_steps=8,
    require_approval=True,
    verbose=True
)

### Step 2.8 - Display Agent Results

In [ ]:
def display_agent_dashboard(result: Dict[str, Any]):
    """Display a formatted agent result dashboard."""
    
    confidence = result.get("confidence", "unknown")
    conf_colors = {"high": "#2e7d32", "medium": "#f57f17", "low": "#c62828"}
    conf_color = conf_colors.get(confidence, "#757575")
    
    approval = result.get("approval_state", "unknown")
    approval_labels = {
        "pending_human_approval": ("Pending Approval", "#e65100"),
        "auto_approved": ("Auto-Approved", "#2e7d32"),
        "incomplete": ("Incomplete", "#c62828")
    }
    approval_text, approval_color = approval_labels.get(approval, (approval, "#757575"))
    
    # Build tool history table
    tool_rows = ""
    for i, th in enumerate(result.get("tool_history", []), 1):
        args_str = ", ".join(f"{k}={v}" for k, v in th.get("args", {}).items())
        tool_rows += f"""
        <tr>
            <td style="padding:8px; border-bottom:1px solid #e0e0e0;">{i}</td>
            <td style="padding:8px; border-bottom:1px solid #e0e0e0;"><code>{th.get('tool', '')}</code></td>
            <td style="padding:8px; border-bottom:1px solid #e0e0e0; font-size:0.9em;">{args_str}</td>
        </tr>
        """
    
    # Build trace table
    type_colors = {"action": "#1565c0", "final": "#2e7d32", "guardrail": "#e65100", "retry": "#c62828"}
    trace_rows = ""
    for t in result.get("trace", []):
        step_type = t.get("type", "")
        color = type_colors.get(step_type, "#757575")
        badge = f'<span style="background:{color};color:#fff;padding:2px 8px;border-radius:4px;font-size:0.8em;">{step_type}</span>'
        tool_cell = f'<code>{t.get("tool", "")}</code>' if t.get("tool") else "-"
        thought = (t.get("thought", "") or t.get("error", ""))[:80]
        
        trace_rows += f"""
        <tr>
            <td style="padding:6px; border-bottom:1px solid #e0e0e0; text-align:center;">{t.get('step', '')}</td>
            <td style="padding:6px; border-bottom:1px solid #e0e0e0;">{badge}</td>
            <td style="padding:6px; border-bottom:1px solid #e0e0e0;">{tool_cell}</td>
            <td style="padding:6px; border-bottom:1px solid #e0e0e0; font-size:0.85em;">{thought}</td>
            <td style="padding:6px; border-bottom:1px solid #e0e0e0; text-align:right;">{t.get('latency_ms', 0):,} ms</td>
        </tr>
        """
    
    answer_html = result.get("answer", "No answer").replace("\n", "<br>")
    
    html = f"""
    <div style="font-family:system-ui,-apple-system,sans-serif; max-width:900px;">
        <div style="background:linear-gradient(135deg,#1a237e 0%,#283593 100%); color:white; 
                    padding:20px; border-radius:8px 8px 0 0;">
            <h2 style="margin:0;">Supply Chain Agent Result</h2>
            <p style="margin:5px 0 0 0; opacity:0.9;">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
        </div>
        
        <div style="border:1px solid #e0e0e0; border-top:none; padding:20px; background:#fafafa;">
            <div style="display:flex; gap:20px; margin-bottom:20px;">
                <div style="background:white; padding:12px 20px; border-radius:6px; border-left:4px solid {conf_color};">
                    <div style="font-size:0.8em; color:#666;">Confidence</div>
                    <div style="font-size:1.4em; font-weight:bold; color:{conf_color};">{confidence.upper()}</div>
                </div>
                <div style="background:white; padding:12px 20px; border-radius:6px; border-left:4px solid {approval_color};">
                    <div style="font-size:0.8em; color:#666;">Status</div>
                    <div style="font-size:1.4em; font-weight:bold; color:{approval_color};">{approval_text}</div>
                </div>
                <div style="background:white; padding:12px 20px; border-radius:6px; border-left:4px solid #1565c0;">
                    <div style="font-size:0.8em; color:#666;">Steps</div>
                    <div style="font-size:1.4em; font-weight:bold;">{result.get('steps_taken', 0)}</div>
                </div>
            </div>
            
            <div style="background:white; padding:15px; border-radius:6px; margin-bottom:15px;">
                <h4 style="margin-top:0;">Agent Recommendation</h4>
                <div style="line-height:1.6;">{answer_html}</div>
            </div>
            
            <div style="background:white; padding:15px; border-radius:6px; margin-bottom:15px;">
                <h4 style="margin-top:0;">Tools Used ({len(result.get('tool_history', []))})</h4>
                <table style="width:100%; border-collapse:collapse;">
                    <thead>
                        <tr style="background:#f5f5f5;">
                            <th style="padding:8px; text-align:left; border-bottom:2px solid #ddd;">#</th>
                            <th style="padding:8px; text-align:left; border-bottom:2px solid #ddd;">Tool</th>
                            <th style="padding:8px; text-align:left; border-bottom:2px solid #ddd;">Arguments</th>
                        </tr>
                    </thead>
                    <tbody>{tool_rows}</tbody>
                </table>
            </div>
            
            <div style="background:white; padding:15px; border-radius:6px;">
                <h4 style="margin-top:0;">Reasoning Trace</h4>
                <table style="width:100%; border-collapse:collapse;">
                    <thead>
                        <tr style="background:#f5f5f5;">
                            <th style="padding:6px; text-align:center; border-bottom:2px solid #ddd;">Step</th>
                            <th style="padding:6px; text-align:left; border-bottom:2px solid #ddd;">Type</th>
                            <th style="padding:6px; text-align:left; border-bottom:2px solid #ddd;">Tool</th>
                            <th style="padding:6px; text-align:left; border-bottom:2px solid #ddd;">Thought</th>
                            <th style="padding:6px; text-align:right; border-bottom:2px solid #ddd;">Latency</th>
                        </tr>
                    </thead>
                    <tbody>{trace_rows}</tbody>
                </table>
            </div>
        </div>
    </div>
    """
    
    display(HTML(html))


# Display the dashboard
display_agent_dashboard(agent_result)

### Step 2.9 - Human-in-the-Loop Approval

For high-stakes decisions, the agent's recommendation requires human approval before action.

In [ ]:
def simulate_approval_workflow(agent_result: Dict[str, Any]) -> Dict[str, Any]:
    """Simulate the human approval workflow."""
    
    print("\n" + "=" * 60)
    print("HUMAN-IN-THE-LOOP APPROVAL GATE")
    print("=" * 60)
    
    print(f"\nAgent Recommendation:")
    print(f"  Confidence: {agent_result.get('confidence', 'unknown')}")
    print(f"  Tools used: {len(agent_result.get('tool_history', []))}")
    print(f"\nSummary: {agent_result.get('answer', '')[:200]}...")
    
    print("\n" + "-" * 60)
    print("In production, this would trigger:")
    print("  1. Notification to Supply Chain Manager")
    print("  2. SAP Workflow approval task")
    print("  3. If approved: auto-create Purchase Requisition")
    print("  4. If rejected: log reason and archive")
    print("-" * 60)
    
    # In workshop, we'll simulate approval
    approval = input("\nApprove this recommendation? (yes/no): ").strip().lower()
    
    if approval in ["yes", "y"]:
        agent_result["approval_state"] = "approved"
        agent_result["approved_by"] = "workshop_user"
        agent_result["approved_at"] = datetime.now().isoformat()
        print("\n[APPROVED] Proceeding to create purchase requisition...")
    else:
        agent_result["approval_state"] = "rejected"
        agent_result["rejected_by"] = "workshop_user"
        agent_result["rejected_at"] = datetime.now().isoformat()
        print("\n[REJECTED] Recommendation archived for review.")
    
    return agent_result


# Run approval workflow (comment out if not interactive)
# agent_result = simulate_approval_workflow(agent_result)

---

## Checkpoint & Exercise

### What You've Learned

1. **Agentic Pattern**: When adaptive reasoning adds value over fixed workflows
2. **Tool Design**: How to expose business capabilities to an LLM
3. **ReAct Loop**: Think → Act → Observe cycle with guardrails
4. **Observability**: Full trace of agent reasoning for audit
5. **Governance**: Human-in-the-loop approval for high-stakes decisions

### Exercise: Try Different Scenarios

Modify the `agent_task` and re-run:

In [ ]:
# Try a different scenario
exercise_task = """
Compare the risk profile of VENDOR_A vs VENDOR_B for a 1500 unit order of Sensors (MAT-1001).
Which vendor would you recommend and why?
"""

exercise_result = run_supply_chain_agent(
    task=exercise_task,
    min_tools=2,
    max_steps=6,
    verbose=True
)

display_agent_dashboard(exercise_result)

### Discussion Questions

1. How would you add a tool that checks real-time logistics data?
2. What guardrails would you add for production deployment?
3. How would you handle conflicting recommendations from different tools?
4. What metrics would you track to measure agent effectiveness?

---

## Summary: Part 2 Complete

You've built an agentic supply chain risk management system that:

| Component | What It Does |
|-----------|-------------|
| **Tools** | Expose business capabilities (risk assessment, alternatives, proposals) |
| **ReAct Loop** | Adaptive reasoning based on discovered information |
| **Guardrails** | Ensure minimum evidence gathering before conclusions |
| **Trace** | Full observability of agent reasoning |
| **Approval Gate** | Human-in-the-loop for high-stakes decisions |

### Architecture Principles Applied

1. **Use agents only when needed** - Part 1 showed deterministic works for single POs
2. **Strict tool contracts** - Clear inputs/outputs for each capability
3. **Evidence-backed decisions** - Agent must use tools, can't fabricate data
4. **Observable by design** - Every step logged for audit
5. **Human oversight for action** - Recommendations, not autonomous execution

### Next: Architecture Discussion

The instructor will now lead a discussion on:
- Productionizing this to a CAP application
- Connecting to live S/4HANA data
- End-to-end BTP architecture

See: `architecture_discussion.md`

---

## Cleanup

In [ ]:
def cleanup_session():
    """Clean up sensitive data at end of workshop."""
    print("Cleaning up workshop session...")
    
    sensitive_vars = ["AICORE_CLIENT_ID", "AICORE_CLIENT_SECRET"]
    for var in sensitive_vars:
        if var in os.environ:
            del os.environ[var]
    print("   [OK] Cleared sensitive environment variables")
    
    print("\nCleanup complete.")

# Uncomment to run cleanup:
# cleanup_session()